# Notebook 4 — Binary-result audits, confounders, overlap check, and *held-out* saliency

**Two source types are necessary:**
1. `Code (2).zip` — your earlier **binary** fold metrics and predictions. This permits window-by-window subject/class counts, CIs, protocol evidence, and cautiously paired reanalysis (but does **not** repair original subject overlap).
2. Executed **Notebook 1 results ZIP**, plus the **original ds004504 EEG dataset** for the optional full 88-subject saliency experiment. This permits independent-subject saliency and subject-level confounder/recording-length analyses.

This notebook will never relabel the original subject-overlapping binary scores as leakage-free. **Actual numeric window-overlap cannot be proved** from old binary CSVs because they lack segment start/end identifiers. It writes that limitation explicitly. It also does not manufacture age, MMSE, preprocessing metadata or significance.

**Recommended order:** Run Notebooks 1 → 2 → 3 → 4. For the saliency section, set `RUN_GROUPED_SALIENCY=True` and upload the original EEG ZIP when requested. Cells run in Colab. Saved data and all tables are packed into a final results ZIP.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib mne eeglabio tensorflow
import io, os, json, re, sys, math, hashlib, zipfile, warnings, gc
from pathlib import Path
from collections import defaultdict
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy import signal,stats
from sklearn.metrics import f1_score,accuracy_score
from IPython.display import display
import tensorflow as tf, mne
from tensorflow.keras import layers,Model
from tensorflow.keras.optimizers.schedules import LearningRateSchedule
TARGET_CHANNELS=["Fp1","Fp2","F7","F3","Fz","F4","F8","T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]
TARGET_SFREQ=256;WINDOW_SECONDS=5;WINDOW_SAMPLES=1280;N_CHANNELS=19;BOUNDARY_BUFFER_SECONDS=.5
warnings.filterwarnings('ignore',category=FutureWarning)
OUTPUT=Path('/content/Notebook4_Binary_Audit_Confounders_Saliency')
OUTPUT.mkdir(parents=True,exist_ok=True)
ORIGINAL_CODE_ZIP=Path('/content/Code (2).zip')
GROUPED_RESULTS_DIR=Path('/content/EEG_AD_FTD_CN_GroupKFold/results')
GROUPED_RESULTS_ZIP=Path('/content/EEG_AD_FTD_CN_GroupKFold/AD_FTD_CN_GroupKFold_Reviewer_Results.zip')
RUN_GROUPED_SALIENCY=True  # switch False to run ONLY archived binary and metadata audits
MAX_WINDOWS_PER_SUBJECT=20  # evenly spaced valid 5s samples; maximum 20 per unseen subject
N_PERMUTATIONS=1500
SUBJECT_BOOT_REPS=1000
RANDOM_SEED=2026
rng=np.random.default_rng(RANDOM_SEED)
print('Working output:',OUTPUT)

def safe_unzip(zippath,dest):
    dest=Path(dest).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zippath) as z:
        for m in z.infolist():
            if not (dest/m.filename).resolve().is_relative_to(dest):raise ValueError('Unsafe ZIP path')
            if (m.external_attr >> 16)&0o170000==0o120000:raise ValueError('Unsupported symlink')
        z.extractall(dest)

def find_or_upload(path,label):
    if path.exists():return path
    from google.colab import files
    print('Upload',label)
    uploaded=files.upload()
    items=[Path('/content')/name for name in uploaded.keys() if name.lower().endswith('.zip')]
    if not items:raise FileNotFoundError('Required ZIP not uploaded: '+label)
    del uploaded
    return items[0]

## A. Audit existing binary fold files — no retraining required

The original CSVs have `fold`, `subject_id`, `y_true`, `pred_tuned`, `prob_disease`, and the chosen threshold. We recover each window setting, 10-fold metrics, subject-recurrence count, and available data to explain the originally published "tuned" numbers. Treat confidence intervals as **descriptive segment-fold intervals**, not subject-generalization intervals.

In [ ]:
ORIGINAL_CODE_ZIP=find_or_upload(ORIGINAL_CODE_ZIP,'the original Code (2).zip of notebooks and binary results')
assert zipfile.is_zipfile(ORIGINAL_CODE_ZIP)
archives={}
with zipfile.ZipFile(ORIGINAL_CODE_ZIP) as z:
    names=set(z.namelist())
    folders=sorted(n.rsplit('/',1)[0] for n in names if n.endswith('/fold_metrics.csv'))
    assert folders, 'No fold_metrics.csv found in old code ZIP'
    for folder in folders:
        fn=folder.rsplit('/',1)[-1]
        prefix=folder+'/'
        if prefix+'segment_predictions.csv' not in names:continue
        folds=pd.read_csv(io.BytesIO(z.read(prefix+'fold_metrics.csv')))
        preds=pd.read_csv(io.BytesIO(z.read(prefix+'segment_predictions.csv')))
        assert {'fold','subject_id','y_true','pred_tuned','prob_disease'}.issubset(preds.columns)
        assert {'fold','tuned_accuracy','tuned_f1','tuned_roc_auc'}.issubset(folds.columns)
        task='FTD-CN' if 'ftd_cn' in fn else 'AD-CN'
        window_match=re.search(r'(5s|10s|15s|20s|25s|30s)(?:_15s_overlap)?',fn)
        window='5s' if fn.startswith('ablation_5s') else ('30s + 15s overlap' if window_match and 'overlap' in window_match.group(0) else (window_match.group(0) if window_match else '?'))
        kind='ablation' if fn.startswith('ablation_') else 'window'
        if kind=='ablation':
            variant=('dual_no_se' if 'without_se_attention' in fn else
                     'freq_only' if 'frequency_domain' in fn else
                     'time_only' if 'time_domain' in fn else 'other')
        else:variant='full_binary'
        assert folds.fold.nunique()==10 and preds.fold.nunique()==10
        archives[fn]={'task':task,'window':window,'kind':kind,'variant':variant,'folds':folds,'preds':preds}
print('Found:',len(archives),'binary experiment folders; saved 10-fold outputs')
if len(archives)<14:print('WARNING: expected at least 14 window-task sets; review the Code ZIP contents.')

In [ ]:
# Tables: class support, subjects, repeated subjects, fold metrics and 95% descriptive CIs
summary=[]; support=[]; fold_class=[]; fold_results=[]
for filename,obj in archives.items():
    p=obj['preds']; f=obj['folds'];task=obj['task'];window=obj['window'];kind=obj['kind'];variant=obj['variant']
    assert p.y_true.isin([0,1]).all()
    subfold=p.groupby('subject_id').fold.nunique()
    summary.append({'experiment':filename,'task':task,'window':window,'kind':kind,'variant':variant,
        'n_segments':len(p),'n_subjects':p.subject_id.nunique(),'folds':f.fold.nunique(),
        'subjects_in_2plus_test_folds':int((subfold>1).sum()),
        'max_folds_per_subject':int(subfold.max()),
        'train_test_independent':'NO: recurring subjects in segment-fold tests'})
    for val,count in p.y_true.value_counts().items():
        support.append({'experiment':filename,'task':task,'window':window,'variant':variant,
                        'label':int(val),'n_segments':int(count),
                        'n_unique_subjects':int(p[p.y_true==val].subject_id.nunique())})
    for (fold,label),g in p.groupby(['fold','y_true']):
        fold_class.append({'experiment':filename,'fold':int(fold),'label':int(label),
                           'n_segments':int(len(g)),'n_subjects':g.subject_id.nunique()})
    for metric in ['tuned_accuracy','tuned_f1','tuned_roc_auc','default_accuracy']:
        v=f[metric].to_numpy(float)
        lo,hi=stats.t.interval(.95,df=len(v)-1,loc=v.mean(),scale=stats.sem(v))
        fold_results.append({'experiment':filename,'task':task,'window':window,
            'variant':variant,'metric':metric,'folds':len(v),'mean':float(v.mean()),
            'SD':float(v.std(ddof=1)),'CI95_low_descriptive':float(lo),
            'CI95_high_descriptive':float(hi),
            'note':'segment-level overlapping subject folds, descriptive only'})
pd.DataFrame(summary).to_csv(OUTPUT/'binary_experiment_index_and_subject_leakage.csv',index=False)
pd.DataFrame(support).to_csv(OUTPUT/'binary_per_window_label_subject_segment_counts.csv',index=False)
pd.DataFrame(fold_class).to_csv(OUTPUT/'binary_per_fold_class_distribution.csv',index=False)
pd.DataFrame(fold_results).to_csv(OUTPUT/'binary_fold_mean_SD_95CI.csv',index=False)
display(pd.DataFrame(summary)[['task','window','kind','variant','n_segments','n_subjects',
                                'subjects_in_2plus_test_folds']])

### B. Original binary ablation — cautious *paired-fold* test

The archive does not have subject-independent test folds. For reproducible paired analysis, we **verify identical fold-by-fold test subject/label counts** before pairing variants; matching counts do not prove identical segment identities, because the files do not include original segment IDs. If the audit does not match, the test is skipped. These statistical comparisons are descriptive and should be distinguished from the new *subject-independent* three-class tests in Notebook 2.

In [ ]:
from scipy.stats import wilcoxon
# 5s original full binary vs time_only, and vs dual_no_se when present
comparisons=[]
for task in ['AD-CN','FTD-CN']:
    cases={}
    for name,o in archives.items():
        if o['task']==task and o['window']=='5s':
            cases[o['variant']]=o
    for a,b in [('full_binary','time_only'),('full_binary','dual_no_se'),('time_only','dual_no_se')]:
        if a not in cases or b not in cases:
            comparisons.append({'task':task,'a':a,'b':b,'test':'NOT AVAILABLE: missing original comparison file'})
            continue
        x,y=cases[a],cases[b]
        def signature(p):
            return p.groupby(['fold','subject_id','y_true']).size().rename('n').reset_index().sort_values(['fold','subject_id','y_true']).reset_index(drop=True)
        equal=signature(x['preds']).equals(signature(y['preds']))
        row={'task':task,'a':a,'b':b,'same_fold_subject_class_counts':bool(equal),
             'has_segment_UIDs':False,'test':'SKIPPED (unverified fold pairing)'}
        if equal:
            a_f=x['folds'].sort_values('fold');b_f=y['folds'].sort_values('fold')
            assert np.array_equal(a_f.fold,b_f.fold)
            # Original values are PERCENT for tuned_accuracy and tuned_f1.
            for metric in ['tuned_accuracy','tuned_f1','tuned_roc_auc']:
                dif=a_f[metric].to_numpy()-b_f[metric].to_numpy()
                row['mean_delta_'+metric]=float(dif.mean())
                row['paired_Wilcoxon_p_'+metric]=float(wilcoxon(dif).pvalue) if np.any(dif!=0) else 1.
            row['test']='EXPLORATORY paired 10-fold Wilcoxon; not subject-independent'
        comparisons.append(row)
pair=pd.DataFrame(comparisons)
pair.to_csv(OUTPUT/'original_binary_ablation_paired_fold_analysis.csv',index=False)
display(pair)

In [ ]:
# Explain what cannot be established about overlapping windows from the old CSV schema.
overlap_files=[name for name,obj in archives.items() if 'overlap' in obj['window']]
rows=[]
for name in overlap_files:
    p=archives[name]['preds']
    has_timestamps={'start_sec','end_sec'}.issubset(p.columns)
    has_segment_id=any(k in p.columns for k in ['segment_uid','window_id','segment_index'])
    rows.append({'experiment':name,'n_segments':len(p),
        'recurring_subjects_across_folds':int((p.groupby('subject_id').fold.nunique()>1).sum()),
        'start_end_times_available':has_timestamps,'segment_ID_available':has_segment_id,
        'direct_cross_fold_temporal_overlap_check_possible':bool(has_timestamps),
        'status':'UNRESOLVED: original archive lacks window-level temporal identifiers' if not has_timestamps
                  else 'CAN CHECK: add timestamp overlap scan'})
overlap_audit=pd.DataFrame(rows)
overlap_audit.to_csv(OUTPUT/'overlap_window_identifiability_audit.csv',index=False)
display(overlap_audit)
# If future recordings include start/end timestamps, a rigorous overlap check is:
def overlapping_train_test_intervals(table, a='train',b='test'):
    required={'participant_id','split','start_sec','end_sec'}
    if not required.issubset(table):raise ValueError('Exact overlap audit requires '+str(required))
    x=table[table.split==a];y=table[table.split==b];events=[]
    for sid,u in x.groupby('participant_id'):
        v=y[y.participant_id==sid]
        for xx in u.itertuples(index=False):
            for yy in v.itertuples(index=False):
                if xx.start_sec<yy.end_sec and yy.start_sec<xx.end_sec:
                    events.append({'subject':sid,'train_start':xx.start_sec,'test_start':yy.start_sec})
    return pd.DataFrame(events)
print('Original archive lacks enough timing data for an honest direct 30s+15s overlap audit.')

## C. Recording duration, age and MMSE confounders

Notebook 1 exports `subject_recording_duration_and_segments.csv`; merge this with `OOF_subject_predictions.csv` and the original ds004504 `participants.tsv`. **Missing columns are reported as missing, not fabricated.** Exploratory associations are at the **subject level (n=88)**. If no metadata file is provided, the recording-duration analysis can still run from Notebook 1 results.

In [ ]:
# Retrieve grouped predictions/results. It is needed for metadata analysis and saliency.
need_grouped=['OOF_subject_predictions.csv','subject_split_manifest.csv','subject_recording_duration_and_segments.csv']
if not all((GROUPED_RESULTS_DIR/p).exists() for p in need_grouped):
    GROUPED_RESULTS_ZIP=find_or_upload(GROUPED_RESULTS_ZIP,'executed Notebook 1 GroupKFold reviewer results ZIP')
    GROUPED_RESULTS_DIR=Path('/content/Notebook4_grouped_results')
    safe_unzip(GROUPED_RESULTS_ZIP,GROUPED_RESULTS_DIR)
    candidates=[p.parent for p in GROUPED_RESULTS_DIR.rglob('OOF_subject_predictions.csv')]
    GROUPED_RESULTS_DIR=next((p for p in candidates if all((p/q).exists() for q in need_grouped)),None)
    if GROUPED_RESULTS_DIR is None:raise FileNotFoundError('Grouped results ZIP missing important Notebook 1 outputs')
sub=pd.read_csv(GROUPED_RESULTS_DIR/'OOF_subject_predictions.csv')
split=pd.read_csv(GROUPED_RESULTS_DIR/'subject_split_manifest.csv')
duration=pd.read_csv(GROUPED_RESULTS_DIR/'subject_recording_duration_and_segments.csv')
assert len(sub)==88 and sub.participant_id.nunique()==88
assert split.query('partition == "test"').participant_id.nunique()==88
meta=sub.merge(duration,on='participant_id',how='left',suffixes=('','_duration'),validate='one_to_one')
assert meta.recording_duration_sec.notna().all()
print('Duration from Notebook 1:');display(meta.groupby('group')['recording_duration_sec'].agg(['count','median','min','max']))

In [ ]:
# If a full saliency run is requested, obtain the EEG ZIP before the metadata analysis
# so age/MMSE can be evaluated on the first notebook run (not only after re-running).
if RUN_GROUPED_SALIENCY:
    preexisting=any(list(p.rglob('participants.tsv')) for p in [Path('/content/openneuro_uploaded_data'),
            Path('/content/ds004504'),Path('/content/openneuro_data'),Path('/content/notebook4_ds004504')] if p.exists())
    if not preexisting and not Path('/content/participants.tsv').exists():
        from google.colab import files
        print('Upload source ds004504 EEG DATASET ZIP for age/MMSE and saliency. Not the Code ZIP.')
        incoming=files.upload()
        zips=[Path('/content')/name for name in incoming if name.lower().endswith('.zip')]
        if not zips:raise FileNotFoundError('No EEG dataset ZIP uploaded')
        extracted=Path('/content/notebook4_ds004504')
        for z in zips:safe_unzip(z,extracted)
        del incoming
# Optional metadata TSV; if raw dataset is already present, try loading it automatically.
MANUAL_PARTICIPANTS_TSV=Path('/content/participants.tsv')
DATA_ROOT_HINTS=[Path('/content/openneuro_uploaded_data'),Path('/content/ds004504'),Path('/content/openneuro_data'),Path('/content/notebook4_ds004504')]
candidates=[MANUAL_PARTICIPANTS_TSV] if MANUAL_PARTICIPANTS_TSV.exists() else []
for rt in DATA_ROOT_HINTS:
    if rt.exists():candidates += list(rt.rglob('participants.tsv'))
source_tsv=next((p for p in candidates if p.exists()),None)
if source_tsv is not None:
    old=pd.read_csv(source_tsv,sep='\t',na_values=['n/a','NA','N/A','nan'])
    old.participant_id=old.participant_id.astype(str).str.strip()
    keep=['participant_id']
    for kind in ['age','mmse']:
        hits=[c for c in old.columns if (kind in c.lower() and c not in keep)]
        if hits:keep.append(hits[0])
    old=old[keep].drop_duplicates('participant_id')
    meta=meta.merge(old,on='participant_id',how='left',validate='one_to_one')
    print('Participants source:',source_tsv,'; found columns:',list(old.columns))
else:print('No participants.tsv found. Age/MMSE analysis will remain unassessed. Add the original TSV and rerun this cell.')

feature_cols=['recording_duration_sec','clean_windows']
feature_cols += [c for c in meta if 'age' in c.lower() or 'mmse' in c.lower()]
feature_cols=list(dict.fromkeys(c for c in feature_cols if c in meta.columns))
confound_rows=[]
for feature in feature_cols:
    v=pd.to_numeric(meta[feature],errors='coerce')
    if v.notna().sum()<4:
        confound_rows.append({'feature':feature,'n_valid':int(v.notna().sum()),'status':'NOT ASSESSABLE'})
        continue
    valid=v.notna() & meta.segment_accuracy.notna()
    rho,p=stats.spearmanr(v[valid],pd.to_numeric(meta.loc[valid,'segment_accuracy'])) if valid.sum()>=4 else (np.nan,np.nan)
    classes=[v[meta.group==g].dropna().values for g in ['HC','AD','FTD']]
    try: kw_p=stats.kruskal(*classes).pvalue if all(len(c)>1 for c in classes) else np.nan
    except ValueError:kw_p=np.nan
    confound_rows.append({'feature':feature,'n_valid':int(v.notna().sum()),
       'n_missing':int(v.isna().sum()),'spearman_rho_with_subject_segment_accuracy':rho,
       'spearman_p_exploratory':p,'Kruskal_group_difference_p_exploratory':kw_p,
       'HC_median':float(np.nanmedian(classes[0])) if len(classes[0]) else np.nan,
       'AD_median':float(np.nanmedian(classes[1])) if len(classes[1]) else np.nan,
       'FTD_median':float(np.nanmedian(classes[2])) if len(classes[2]) else np.nan,
       'status':'EXPLORATORY: possible confounding, no causal inference'})
meta.to_csv(OUTPUT/'subject_level_metadata_duration_performance.csv',index=False)
confound_table=pd.DataFrame(confound_rows)
confound_table.to_csv(OUTPUT/'recording_age_MMSE_confounder_audit.csv',index=False)
display(confound_table)

## D. Independent-subject channel saliency and permutation tests

**Additional computation on new model checkpoints.** This section runs saliency for the *outer held-out subjects* only. It reloads each fold's exact Notebook 1 checkpoint and its **training-only** Welch/biomarker normalizers. Every subject contributes up to 20 evenly spaced boundary-clean 5-second windows (equal subject weighting). Attribution is **input × gradient**, not claimed to be the original Grad-CAM implementation. Explain the difference in the manuscript. Two exploratory null tests: (i) subject-level random channel-label permutations for the previously highlighted O1/O2/T5/T6 region and (ii) permutation of diagnosis labels across subjects for each channel, with Benjamini–Hochberg FDR correction. Neither is a clinical biomarker validation.

**Needs original dataset ZIP in this runtime.** If it is unavailable, switch `RUN_GROUPED_SALIENCY=False` and complete the binary/metadata audits first; saliency will remain incomplete until executed.

In [ ]:
if RUN_GROUPED_SALIENCY:
    # TensorFlow is installed by the top-level package cell.
    import tensorflow as tf
    from tensorflow.keras import layers,Model
    import mne
    mne.set_log_level('ERROR')
    TARGET_SFREQ=256;WINDOW_SECONDS=5;WINDOW_SAMPLES=1280
    TARGET_CHANNELS=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
    N_CHANNELS=19; BOUNDARY_BUFFER_SECONDS=.5
    CLASS_NAMES=['HC','AD','FTD']
    # Find already-extracted dataset; if missing, request user upload.
    def find_root(root):
        root=Path(root)
        if not root.exists():return None
        for p in ([root] if (root/'participants.tsv').exists() else []) + list(root.rglob('participants.tsv')):
            q=p if p.is_dir() else p.parent
            if (q/'participants.tsv').exists() and (q/'derivatives').exists():return q
        return None
    DATA_ROOT=next((q for r in DATA_ROOT_HINTS if (q:=find_root(r)) is not None),None)
    if DATA_ROOT is None:
        from google.colab import files
        print('Upload the original ds004504 DATASET ZIP (contains participants.tsv and derivatives/)')
        incoming=files.upload()
        zipps=[Path('/content')/p for p in incoming if p.lower().endswith('.zip')]
        if not zipps:raise FileNotFoundError('Required source EEG dataset ZIP missing')
        extracted=Path('/content/notebook4_ds004504')
        for z in zipps:safe_unzip(z,extracted)
        DATA_ROOT=find_root(extracted)
        if DATA_ROOT is None:raise FileNotFoundError('No derivatives/ and participants.tsv found after upload')
    PARTICIPANTS_PATH=DATA_ROOT/'participants.tsv'
    DERIVATIVES_ROOT=DATA_ROOT/'derivatives'
    assert PARTICIPANTS_PATH.exists()
    print('EEG root',DATA_ROOT,'GPU:',tf.config.list_physical_devices('GPU'))
else:
    print('SALIENCY SKIPPED — set RUN_GROUPED_SALIENCY=True and provide dataset + checkpoints to complete it.')

In [ ]:
GROUP_MAP = {
    "A": "AD", "AD": "AD", "ALZHEIMER": "AD", "ALZHEIMERS": "AD",
    "C": "HC", "CN": "HC", "HC": "HC", "CONTROL": "HC",
    "HEALTHY": "HC", "COGNITIVELY NORMAL": "HC",
    "F": "FTD", "FTD": "FTD", "FRONTOTEMPORAL DEMENTIA": "FTD",
}

CHANNEL_ALIASES = {
    "FP1": "Fp1", "FP2": "Fp2", "F7": "F7", "F3": "F3",
    "FZ": "Fz", "F4": "F4", "F8": "F8",
    "T3": "T3", "T7": "T3", "C3": "C3", "CZ": "Cz",
    "C4": "C4", "T4": "T4", "T8": "T4",
    "T5": "T5", "P7": "T5", "P3": "P3", "PZ": "Pz",
    "P4": "P4", "T6": "T6", "P8": "T6", "O1": "O1", "O2": "O2",
}

def normalize_group(value):
    text = str(value).strip().upper().replace("_", " ").replace("-", " ")
    text = " ".join(text.split())
    if text not in GROUP_MAP:
        raise ValueError(f"Unknown group value: {value!r}")
    return GROUP_MAP[text]


def load_participants():
    participants = pd.read_csv(
        PARTICIPANTS_PATH,
        sep="\t",
        na_values=["n/a", "N/A", "NA", "nan", ""],
    )
    participants["participant_id"] = participants["participant_id"].astype(str).str.strip()

    group_column = next(
        (c for c in participants.columns if c.lower() in {"group", "diagnosis", "label"}),
        None,
    )
    if group_column is None:
        raise RuntimeError("No group/diagnosis column found.")

    participants["group"] = participants[group_column].apply(normalize_group)
    participants["label"] = participants["group"].map(LABEL_MAP).astype(np.int32)
    participants = participants[["participant_id", "group", "label"]].copy()

    observed = participants["group"].value_counts().to_dict()
    for group_name, expected in EXPECTED_SUBJECT_COUNTS.items():
        assert int(observed.get(group_name, 0)) == expected
    assert len(participants) == EXPECTED_TOTAL_SUBJECTS
    assert participants["participant_id"].is_unique
    return participants

def canonicalize_channels(raw):
    renames = {}
    for channel in raw.ch_names:
        key = str(channel).strip().upper().replace(" ", "")
        canonical = CHANNEL_ALIASES.get(key, channel)
        if canonical != channel:
            renames[channel] = canonical
    if renames:
        raw.rename_channels(renames)
    return raw

def find_derivative_file(participant_id):
    eeg_dir = DERIVATIVES_ROOT / participant_id / "eeg"
    preferred = eeg_dir / f"{participant_id}_task-eyesclosed_eeg.set"
    if preferred.exists():
        return preferred
    matches = sorted(eeg_dir.glob("*task-eyesclosed*_eeg.set"))
    if matches:
        return matches[0] # Fixed Bug 1: Explicitly return file path element
    raise FileNotFoundError(f"No derivative EEG file for {participant_id}")

def get_boundary_intervals(raw, buffer_seconds=0.5):
    duration = raw.n_times / float(raw.info["sfreq"])
    intervals = []
    for onset, annotation_duration, description in zip(
        raw.annotations.onset,
        raw.annotations.duration,
        raw.annotations.description,
    ):
        if "boundary" not in str(description).lower():
            continue
        relative_onset = float(onset) - float(raw.first_time)
        if relative_onset < -1.0 or relative_onset > duration + 1.0:
            relative_onset = float(onset)
        start = max(0.0, relative_onset - buffer_seconds)
        end = min(
            duration,
            relative_onset + max(0.0, float(annotation_duration)) + buffer_seconds,
        )
        if end <= start:
            end = min(duration, start + 2.0 * buffer_seconds)
        intervals.append((start, end))
    return sorted(intervals)

def segment_overlaps_boundary(start_seconds, end_seconds, intervals):
    return any(
        start_seconds < boundary_end and end_seconds > boundary_start
        for boundary_start, boundary_end in intervals
    )

def load_and_prepare_subject(participant_id):
    file_path = find_derivative_file(participant_id)
    raw = mne.io.read_raw_eeglab(
        str(file_path),
        preload=True,
        verbose=False,
    )
    original_sfreq = float(raw.info["sfreq"])
    boundary_intervals = get_boundary_intervals(
        raw,
        buffer_seconds=BOUNDARY_BUFFER_SECONDS,
    )
    raw = canonicalize_channels(raw)
    missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]
    if missing:
        raise RuntimeError(f"{participant_id}: missing channels {missing}")
    raw.pick(TARGET_CHANNELS)
    raw.reorder_channels(TARGET_CHANNELS)
    if int(round(raw.info["sfreq"])) != TARGET_SFREQ:
        raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
    data_uv = (raw.get_data().T * 1e6).astype(np.float32)

    # Fixed Bug 2: Evaluate 2D index shape column count correctly
    if data_uv.shape[1] != N_CHANNELS or not np.isfinite(data_uv).all():
        raise RuntimeError(f"{participant_id}: invalid prepared EEG array")
    return data_uv, boundary_intervals, file_path, original_sfreq

def calculate_segment_quality(segment_uv):
    channel_ptp = np.ptp(segment_uv, axis=0)
    channel_std = np.std(segment_uv, axis=0)
    return {
        "max_abs_uv": float(np.max(np.abs(segment_uv))),
        "max_ptp_uv": float(np.max(channel_ptp)),
        "median_ptp_uv": float(np.median(channel_ptp)),
        "min_channel_std_uv": float(np.min(channel_std)),
        "median_channel_std_uv": float(np.median(channel_std)),
    }

In [ ]:
# 5 — Feature parameters (matching original multiclass notebook)
WELCH_WINDOW_SAMPLES=512; WELCH_OVERLAP_SAMPLES=256; WELCH_NFFT=512
WELCH_FMIN=1.0; WELCH_FMAX=45.0; WELCH_N_FREQUENCIES=89
N_BIOMARKERS=10; FEATURE_BUILD_BATCH_SIZE=32; NORMALIZATION_BATCH_SIZE=256
WELCH_CLIP_VALUE=5.0; BIOMARKER_CLIP_VALUE=5.0; PSD_EPSILON=1e-12; STD_EPSILON=1e-6
POSTERIOR_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['T5','P3','Pz','P4','T6','O1','O2']])
ANTERIOR_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['Fp1','Fp2','F7','F3','Fz','F4','F8']])
OCCIPITAL_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['O1','O2']])
TEMPORAL_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['T3','T4','T5','T6']])
BIOMARKER_NAMES=['posterior_relative_alpha','log_posterior_anterior_alpha_ratio',
 'posterior_alpha_peak_hz','log_global_theta_alpha_ratio','log_global_slowing_ratio',
 'frontal_relative_theta','occipital_relative_alpha','temporal_relative_alpha',
 'log_global_total_power','log_posterior_total_power']

In [ ]:
def integrate_band(psd, frequencies, low, high, include_high=False):
    mask = (
        (frequencies >= low) & (frequencies <= high)
        if include_high
        else (frequencies >= low) & (frequencies < high)
    )
    if mask.sum() < 2:
        raise RuntimeError(f"Insufficient bins for {low}-{high} Hz")
    if hasattr(np, "trapezoid"):
        return np.trapezoid(psd[:, mask, :], frequencies[mask], axis=1)
    return np.trapz(psd[:, mask, :], frequencies[mask], axis=1)


def compute_features(segments_uv):
    segments_uv = np.asarray(segments_uv, dtype=np.float32)
    assert segments_uv.shape[1:] == (WINDOW_SAMPLES, N_CHANNELS)

    frequencies, psd = signal.welch(
        segments_uv,
        fs=TARGET_SFREQ,
        window="hann",
        nperseg=WELCH_WINDOW_SAMPLES,
        noverlap=WELCH_OVERLAP_SAMPLES,
        nfft=WELCH_NFFT,
        detrend="constant",
        return_onesided=True,
        scaling="density",
        axis=1,
        average="mean",
    )

    mask = (frequencies >= WELCH_FMIN) & (frequencies <= WELCH_FMAX)
    frequencies = frequencies[mask].astype(np.float32)
    psd = psd[:, mask, :].astype(np.float64)

    assert len(frequencies) == WELCH_N_FREQUENCIES

    log_abs = np.log(psd + PSD_EPSILON)
    rel_psd = psd / (np.sum(psd, axis=1, keepdims=True) + PSD_EPSILON)
    log_rel = np.log(rel_psd + PSD_EPSILON)
    welch_maps = np.stack([log_abs, log_rel], axis=-1).astype(np.float32)

    delta = integrate_band(psd, frequencies, 1.0, 4.0)
    theta = integrate_band(psd, frequencies, 4.0, 8.0)
    alpha = integrate_band(psd, frequencies, 8.0, 13.0)
    beta = integrate_band(psd, frequencies, 13.0, 30.0)
    total = integrate_band(psd, frequencies, 1.0, 45.0, include_high=True)

    rel_theta = theta / (total + PSD_EPSILON)
    rel_alpha = alpha / (total + PSD_EPSILON)

    posterior_alpha = np.mean(alpha[:, POSTERIOR_INDICES], axis=1)
    anterior_alpha = np.mean(alpha[:, ANTERIOR_INDICES], axis=1)

    posterior_psd = np.mean(psd[:, :, POSTERIOR_INDICES], axis=2)
    peak_mask = (frequencies >= 7.0) & (frequencies <= 13.0)
    peak_frequencies = frequencies[peak_mask]
    peak_hz = peak_frequencies[np.argmax(posterior_psd[:, peak_mask], axis=1)]

    global_delta = np.mean(delta, axis=1)
    global_theta = np.mean(theta, axis=1)
    global_alpha = np.mean(alpha, axis=1)
    global_beta = np.mean(beta, axis=1)
    global_total = np.mean(total, axis=1)
    posterior_total = np.mean(total[:, POSTERIOR_INDICES], axis=1)

    biomarkers = np.column_stack(
        [
            np.mean(rel_alpha[:, POSTERIOR_INDICES], axis=1),
            np.log((posterior_alpha + PSD_EPSILON) / (anterior_alpha + PSD_EPSILON)),
            peak_hz,
            np.log((global_theta + PSD_EPSILON) / (global_alpha + PSD_EPSILON)),
            np.log(
                (global_delta + global_theta + PSD_EPSILON)
                / (global_alpha + global_beta + PSD_EPSILON)
            ),
            np.mean(rel_theta[:, ANTERIOR_INDICES], axis=1),
            np.mean(rel_alpha[:, OCCIPITAL_INDICES], axis=1),
            np.mean(rel_alpha[:, TEMPORAL_INDICES], axis=1),
            np.log(global_total + PSD_EPSILON),
            np.log(posterior_total + PSD_EPSILON),
        ]
    ).astype(np.float32)

    assert np.isfinite(welch_maps).all()
    assert np.isfinite(biomarkers).all()
    return welch_maps, biomarkers, frequencies


def training_stats(array, indices, batch_size=256):
    shape = array.shape[1:]
    total = np.zeros(shape, dtype=np.float64)
    total_sq = np.zeros(shape, dtype=np.float64)
    count = 0

    for start in range(0, len(indices), batch_size):
        batch = np.asarray(array[indices[start:start + batch_size]], dtype=np.float64)
        total += np.sum(batch, axis=0)
        total_sq += np.sum(np.square(batch), axis=0)
        count += len(batch)

    mean = total / count
    variance = np.maximum(total_sq / count - np.square(mean), STD_EPSILON ** 2)
    return mean.astype(np.float32), np.sqrt(variance).astype(np.float32)

In [ ]:
if not hasattr(layers, "GroupNormalization"):
    raise RuntimeError("Current Keras GroupNormalization is required.")

@tf.keras.utils.register_keras_serializable(package="EEG")
class ChannelSegmentZScoreClip(layers.Layer):
    def __init__(self, clip_value=6.0, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.clip_value = float(clip_value)
        self.epsilon = float(epsilon)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=1, keepdims=True)
        variance = tf.reduce_mean(tf.square(inputs - mean), axis=1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return tf.clip_by_value(normalized, -self.clip_value, self.clip_value)

    def get_config(self):
        config = super().get_config()
        config.update({"clip_value": self.clip_value, "epsilon": self.epsilon})
        return config


@tf.keras.utils.register_keras_serializable(package="EEG")
class SparseLabelSmoothedCrossentropy(tf.keras.losses.Loss):
    def __init__(self, num_classes=3, label_smoothing=0.02, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = int(num_classes)
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        one_hot = tf.one_hot(y_true, depth=self.num_classes)
        return tf.keras.losses.categorical_crossentropy(
            one_hot,
            y_pred,
            label_smoothing=self.label_smoothing,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {"num_classes": self.num_classes, "label_smoothing": self.label_smoothing}
        )
        return config


@tf.keras.utils.register_keras_serializable(package="EEG")
class WarmupCosineDecay(LearningRateSchedule):
    def __init__(self, peak_lr, min_lr, warmup_steps, total_steps, **kwargs):
        super().__init__()
        self.peak_lr = float(peak_lr)
        self.min_lr = float(min_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps = int(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(max(self.warmup_steps, 1), tf.float32)
        total_steps = tf.cast(max(self.total_steps, self.warmup_steps + 1), tf.float32)
        warmup_lr = self.peak_lr * step / warmup_steps
        progress = tf.clip_by_value(
            (step - warmup_steps) / (total_steps - warmup_steps),
            0.0,
            1.0,
        )
        cosine_lr = self.min_lr + (self.peak_lr - self.min_lr) * 0.5 * (
            1.0 + tf.cos(tf.constant(np.pi, tf.float32) * progress)
        )
        return tf.where(step < warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "peak_lr": self.peak_lr,
            "min_lr": self.min_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
        }


def se_block(x, ratio, prefix):
    channels = int(x.shape[-1])
    a = layers.GlobalAveragePooling2D(name=f"{prefix}_gap")(x)
    a = layers.Dense(max(channels // ratio, 8), activation="gelu", name=f"{prefix}_fc1")(a)
    a = layers.Dense(channels, activation="sigmoid", name=f"{prefix}_fc2")(a)
    a = layers.Reshape((1, 1, channels), name=f"{prefix}_reshape")(a)
    return layers.Multiply(name=f"{prefix}_scale")([x, a])


def cgn(x, filters, kernel, groups, name, separable=False):
    conv = layers.SeparableConv2D if separable else layers.Conv2D
    x = conv(
        filters,
        kernel,
        padding="same",
        use_bias=False,
        name=f"{name}_conv",
    )(x)
    x = layers.GroupNormalization(
        groups=groups,
        axis=-1,
        epsilon=1e-5,
        name=f"{name}_gn",
    )(x)
    return layers.Activation("elu", name=f"{name}_elu")(x)


def build_model():
    raw_input = layers.Input((1280, 19, 1), name="raw_eeg_microvolts")
    welch_input = layers.Input((89, 19, 2), name="welch_absolute_relative_psd")
    biomarker_input = layers.Input((10,), name="spectral_biomarkers")

    # Temporal branch
    t0 = ChannelSegmentZScoreClip(name="temporal_zscore")(raw_input)
    t1 = cgn(t0, 32, (32, 1), 8, "time32")
    t2 = cgn(t0, 32, (64, 1), 8, "time64")
    t3 = cgn(t0, 32, (128, 1), 8, "time128")
    t = layers.Concatenate(name="time_concat")([t1, t2, t3])
    t = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="time_spatial")(t)
    t = layers.GroupNormalization(groups=16, name="time_spatial_gn")(t)
    t = layers.Activation("elu")(t)
    t = se_block(t, 8, "time_se1")
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.15)(t)
    t = cgn(t, 192, (16, 1), 16, "time_sep1", separable=True)
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.20)(t)
    t = cgn(t, 256, (8, 1), 16, "time_sep2", separable=True)
    t = se_block(t, 8, "time_se2")
    t = layers.AveragePooling2D((2, 1))(t)
    t = layers.SpatialDropout2D(0.25)(t)
    t = layers.GlobalAveragePooling2D(name="time_embedding")(t)

    # Welch branch
    f = cgn(welch_input, 32, (7, 1), 8, "freq1")
    f = cgn(f, 48, (5, 1), 8, "freq2")
    f = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="freq_spatial")(f)
    f = layers.GroupNormalization(groups=16, name="freq_spatial_gn")(f)
    f = layers.Activation("elu")(f)
    f = se_block(f, 8, "freq_se1")
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.15)(f)
    f = cgn(f, 128, (5, 1), 16, "freq_sep1", separable=True)
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.20)(f)
    f = cgn(f, 192, (3, 1), 16, "freq_sep2", separable=True)
    f = se_block(f, 8, "freq_se2")
    f = layers.SpatialDropout2D(0.25)(f)
    f = layers.GlobalAveragePooling2D(name="freq_embedding")(f)

    # Biomarkers
    b = layers.Dense(32, activation="gelu", name="bio_dense1")(biomarker_input)
    b = layers.Dropout(0.15)(b)
    b = layers.Dense(32, activation="gelu", name="bio_embedding")(b)

    # Fusion
    z = layers.Concatenate(name="fusion")([t, f, b])
    z = layers.Dense(256, activation="gelu", name="fusion_dense1")(z)
    z = layers.LayerNormalization(name="fusion_ln1")(z)
    z = layers.Dropout(0.30)(z)
    z = layers.Dense(96, activation="gelu", name="fusion_dense2")(z)
    z = layers.LayerNormalization(name="fusion_ln2")(z)
    z = layers.Dropout(0.20)(z)
    output = layers.Dense(3, activation="softmax", name="multiclass_output")(z)

    return Model(
        inputs={
            "raw_eeg_microvolts": raw_input,
            "welch_absolute_relative_psd": welch_input,
            "spectral_biomarkers": biomarker_input,
        },
        outputs=output,
    )

In [ ]:
# Fold checkpoint existence, normalization provenance, and subject-disjointity audit.
if RUN_GROUPED_SALIENCY:
    for fold in sorted(split.fold.unique()):
        d=GROUPED_RESULTS_DIR/'folds'/f'fold_{int(fold):02d}'
        for name in ['normalization.npz','best.weights.h5','COMPLETE.json']:
            assert (d/name).exists(),f'Missing trained Notebook 1 fold artifact: {d/name}'
        manifest=json.loads((d/'COMPLETE.json').read_text())
        assert manifest['fold']==int(fold)
        npz=np.load(d/'normalization.npz')
        test_ids=set(split.query('fold == @fold and partition == "test"').participant_id)
        train_ids=set(split.query('fold == @fold and partition == "train"').participant_id)
        valid_ids=set(split.query('fold == @fold and partition == "validation"').participant_id)
        assert test_ids.isdisjoint(train_ids|valid_ids)
        assert train_ids==set(npz['train_subject_ids'].astype(str))
        npz.close()
    print('PASS: grouped weights + fold-specific training normalization and zero subject overlap')

In [ ]:
# Main saliency experiment: each held-out subject evaluated ONLY by its own fold's trained model.
# Input×gradient is one particular model attribution and may differ from Grad-CAM.
if RUN_GROUPED_SALIENCY:
    subjects=split[split.partition=='test'][['fold','participant_id','group','label']].sort_values(['fold','participant_id'])
    assert len(subjects)==88 and subjects.participant_id.nunique()==88
    saliency_rows=[];quality_rows=[]
    for fold,group_df in subjects.groupby('fold',sort=True):
        fold=int(fold);location=GROUPED_RESULTS_DIR/'folds'/f'fold_{fold:02d}'
        tf.keras.backend.clear_session();model=build_model()
        model.load_weights(str(location/'best.weights.h5'))
        with np.load(location/'normalization.npz') as norm:
            statistics={name:norm[name].copy() for name in ('welch_mean','welch_std','bio_mean','bio_std')}
        print('Attribution fold:',fold,'test subjects',len(group_df))
        for _,row in group_df.iterrows():
            sid=row.participant_id;truth=int(row.label)
            data,boundaries,path,original_hz=load_and_prepare_subject(sid)
            n_all=data.shape[0]//WINDOW_SAMPLES
            eligible=np.array([i for i in range(n_all)
                if not segment_overlaps_boundary(i*WINDOW_SECONDS,(i+1)*WINDOW_SECONDS,boundaries)],dtype=int)
            if len(eligible)==0:raise ValueError(f'No eligible boundary-clean windows: {sid}')
            selected=eligible[np.unique(np.round(np.linspace(0,len(eligible)-1,
                num=min(MAX_WINDOWS_PER_SUBJECT,len(eligible)))).astype(int))]
            eeg=np.stack([data[j*WINDOW_SAMPLES:(j+1)*WINDOW_SAMPLES] for j in selected]).astype(np.float32)
            welch,biom,_=compute_features(eeg)
            w=np.clip((welch-statistics['welch_mean'])/statistics['welch_std'],-WELCH_CLIP_VALUE,WELCH_CLIP_VALUE)
            b=np.clip((biom-statistics['bio_mean'])/statistics['bio_std'],-BIOMARKER_CLIP_VALUE,BIOMARKER_CLIP_VALUE)
            a=tf.convert_to_tensor(eeg[...,None],dtype=tf.float32)
            with tf.GradientTape() as tape:
                tape.watch(a)
                probs=model({'raw_eeg_microvolts':a,
                             'welch_absolute_relative_psd':tf.convert_to_tensor(w),
                             'spectral_biomarkers':tf.convert_to_tensor(b)},training=False)
                # True-diagnosis softmax log-score; no label-based model fitting.
                indices=tf.stack([tf.range(len(eeg)),tf.fill([len(eeg)],truth)],axis=1)
                focus=tf.gather_nd(probs,indices)
                score=tf.reduce_sum(tf.math.log(tf.maximum(focus,1e-8)))
            grad=tape.gradient(score,a)
            if grad is None:raise RuntimeError('No input gradient for '+sid)
            raw_mass=tf.reduce_mean(tf.abs(grad*a),axis=[1,3]).numpy()
            raw_mass=raw_mass/(raw_mass.sum(axis=1,keepdims=True)+1e-12)
            mass=raw_mass.mean(axis=0)
            assert np.all(np.isfinite(mass)) and np.isclose(mass.sum(),1,atol=.0001)
            saliency_rows.append({'fold':fold,'participant_id':sid,'group':row.group,'label':truth,
                'n_windows':len(selected),'avg_true_prob':float(probs.numpy()[:,truth].mean()),
                **{c:float(mass[k]) for k,c in enumerate(TARGET_CHANNELS)}})
            quality_rows.append({'participant_id':sid,'group':row.group,'sampled_windows':len(selected),
                'eligible_windows':len(eligible),'recording_seconds':float(len(data)/TARGET_SFREQ),
                'source_file_name':path.name,'original_hz':original_hz})
            del data,eeg,a,probs,grad,welch,biom,w,b
            gc.collect()
        del model;tf.keras.backend.clear_session();gc.collect()
    sal=pd.DataFrame(saliency_rows)
    assert len(sal)==88 and sal.participant_id.nunique()==88
    assert sal.groupby('group').size().to_dict()=={'HC':29,'AD':36,'FTD':23}
    sal.to_csv(OUTPUT/'grouped_heldout_subject_gradient_times_input_saliency.csv',index=False)
    pd.DataFrame(quality_rows).to_csv(OUTPUT/'saliency_sampled_window_audit.csv',index=False)
    print('PASS: 88 independent subject-level attribution vectors')

In [ ]:
# Permutation significance tests (exploratory; prior selection of O1/O2/T5/T6 is NOT held-out discovery).
if RUN_GROUPED_SALIENCY:
    channels=TARGET_CHANNELS
    values=sal[channels].to_numpy(float)
    n_sub,n_ch=values.shape
    idx=np.array([channels.index(x) for x in ['O1','O2','T5','T6']])
    posterior_mass=values[:,idx].sum(axis=1)
    obs=float(posterior_mass.mean())
    # Null exchanges channel LABELS within each subject; EEG channel exchangeability is a heuristic null.
    null=np.empty(N_PERMUTATIONS)
    for k in range(N_PERMUTATIONS):
        rnd=np.array([rng.permutation(n_ch) for _ in range(n_sub)])
        null[k]=np.take_along_axis(values,rnd,axis=1)[:,idx].sum(axis=1).mean()
    posterior_p=(1+int((null>=obs).sum()))/(1+len(null))
    labels=sal.label.to_numpy(int)
    means=np.stack([values[labels==g].mean(axis=0) for g in [0,1,2]],axis=0)
    observed_channel_gap=means.max(axis=0)-means.min(axis=0)
    exceed=np.zeros(n_ch,int)
    for _ in range(N_PERMUTATIONS):
        shuffle=rng.permutation(labels)
        null_means=np.stack([values[shuffle==g].mean(axis=0) for g in [0,1,2]],axis=0)
        exceed+=((null_means.max(axis=0)-null_means.min(axis=0))>=observed_channel_gap)
    p_values=(exceed+1)/(N_PERMUTATIONS+1)
    # Benjamini-Hochberg adjusted q-values
    ranks=np.argsort(p_values);sorted_p=p_values[ranks]
    q_sorted=np.minimum.accumulate((sorted_p*n_ch/(np.arange(n_ch)+1))[::-1])[::-1]
    q=np.empty(n_ch);q[ranks]=np.clip(q_sorted,0,1)
    table=pd.DataFrame({'channel':channels,'mean_importance_all_subjects':values.mean(0),
        'group_max_minus_min_importance':observed_channel_gap,
        'diagnosis_label_permutation_p':p_values,'BH_FDR_q':q})
    for j,g in enumerate(['HC','AD','FTD']):table['mean_'+g]=means[j]
    table=table.sort_values('mean_importance_all_subjects',ascending=False)
    table.to_csv(OUTPUT/'heldout_channel_saliency_class_differences_FDR.csv',index=False)
    pd.DataFrame([{'region':'O1+O2+T5+T6','n_subjects':n_sub,
        'observed_mean_region_mass':obs,'equal_channel_reference_mass':len(idx)/n_ch,
        'channel_label_permutation_p_one_sided':posterior_p,'permutation_reps':N_PERMUTATIONS,
        'note':'exploratory channel-label null; channels not physiologically exchangeable'}]).to_csv(
            OUTPUT/'posterior_temporal_channel_chance_permutation_test.csv',index=False)
    display(table)
    print('Posterior temporal region observed',round(obs,5),'equal-channel baseline',round(4/19,5),
          'exploratory permutation p',round(posterior_p,5))
    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(np.arange(n_ch),values.mean(axis=0))
    ax.axhline(1/n_ch,ls='--',label='Uniform reference 1/19')
    ax.set_xticks(np.arange(n_ch),channels,rotation=45)
    ax.set(ylabel='Mean normalized input×gradient mass',title='Held-out-subject saliency (each subject weighted equally)')
    ax.legend();fig.tight_layout();fig.savefig(OUTPUT/'heldout_channel_saliency_mean.png',dpi=200);plt.show()

In [ ]:
# Export all finished numerical results, with explicit completion flags.
flags={'original_binary_fold_audit':True,
       'original_binary_paired_fold_analysis':True,
       'original_overlap_intersection_proved':False,
       'duration_audit':True,
       'age_MMSE_complete':bool(any('age' in f.lower() and meta[f].notna().any() for f in meta.columns) and
                                 any('mmse' in f.lower() and meta[f].notna().any() for f in meta.columns)),
       'grouped_heldout_saliency_done':bool(RUN_GROUPED_SALIENCY and (OUTPUT/'grouped_heldout_subject_gradient_times_input_saliency.csv').exists()),
       'original_manuscript_GradCAM_reproduced':False,
       'validation_scope':'new saliency: independent held-out subjects; original binary: subject-overlapping'}
(OUTPUT/'analysis_completion_flags.json').write_text(json.dumps(flags,indent=2))
archive=Path('/content/Notebook4_Binary_Confounder_Saliency_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():z.write(p,arcname=p.name)
print('Results ZIP:',archive,'MB',round(archive.stat().st_size/1e6,2))
print('Completion flags:');display(pd.DataFrame([flags]).T)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass

### Scope and unresolved reviewer requests

- The **binary fold/threshold procedure** still requires precise method prose from the historical notebooks; reconstructed fold statistics do not prove what hyperparameter search was used if unspecified.
- Original 30s+15s **direct temporal overlap** cannot be numerically tested from `segment_predictions.csv` alone; original segmentation code should be checked and corrected for a new grouped experiment if needed.
- The new **input×gradient** saliency tests are exploratory and are not identical to original Grad-CAM. With only 88 unique subjects and spatially correlated sensors, permutation channel labels are a simplified null, **not proof of disease-specific biomarkers**.
- To assess age/MMSE, include the dataset's actual `participants.tsv`. Missing values are reported; do not replace them with synthetic data.
- Preprocessing filtering, re-reference and artifact handling should be sourced from the actual derivative-generation documentation, not assumed.
- **No manuscript revisions or experiments are scientifically complete until notebook cells have been executed and the outputs checked**; this notebook implements the analyses, rather than claiming new outcomes.